# Surface-anchored `ConditionalCorrelationSurface` on a plain BlockGraph

All cubes are plain `ZXZ` — no `ConditionalLeafCubeKind` anywhere. The condition is just a `CorrelationSurface` describing a past pipe measurement, and the conditional observable's per-resolution shape decomposes as

$$ O(a) = \text{shared} \;\oplus\; a \cdot \Delta $$

Emission produces shared `OBSERVABLE_INCLUDE(0)` on the trunk plus one `IF(rec_i) { OBSERVABLE_INCLUDE(0) Δ }` (no ELSE).

In [1]:
from tqec.compile.compile import compile_block_graph
from tqec.computation.block_graph import BlockGraph
from tqec.computation.correlation import (
    ConditionalCorrelationSurface,
    CorrelationSurface,
    ZXEdge,
    ZXNode,
)
from tqec.utils.enums import Basis
from tqec.utils.position import Position3D

In [2]:
b1 = Position3D(0, 0, 0)
b2 = Position3D(0, 0, 1)
b3 = Position3D(0, 0, 2)
c = Position3D(1, 0, 2)

In [3]:
def build_graph() -> BlockGraph:
    g = BlockGraph("surface_anchored_demo")
    g.add_cube(b1, "ZXZ")
    g.add_cube(b2, "ZXZ")
    g.add_cube(b3, "ZXZ")
    g.add_cube(c, "ZXZ")
    g.add_pipe(b1, b2)
    g.add_pipe(b2, b3)
    g.add_pipe(b3, c)
    return g


g = build_graph()
g.view_as_html()

In [4]:
# Condition: past pipe measurement on the vertical b1↔b2 pipe (X-basis).
# anchor_z is derived automatically as max(condition span z) + 1 = 2.
_cond = CorrelationSurface(
    span=frozenset({ZXEdge(u=ZXNode(b1, Basis.X), v=ZXNode(b2, Basis.X))})
)

In [5]:
# branch False: just the vertical spine.
_obs_off = CorrelationSurface(
    span=frozenset(
        {
            ZXEdge(u=ZXNode(b1, Basis.Z), v=ZXNode(b2, Basis.Z)),
            ZXEdge(u=ZXNode(b2, Basis.Z), v=ZXNode(b3, Basis.Z)),
        }
    )
)
# branch True: spine + spatial hop into c.
_obs_on = CorrelationSurface(
    span=frozenset(
        {
            ZXEdge(u=ZXNode(b1, Basis.Z), v=ZXNode(b2, Basis.Z)),
            ZXEdge(u=ZXNode(b2, Basis.Z), v=ZXNode(b3, Basis.Z)),
            ZXEdge(u=ZXNode(b3, Basis.Z), v=ZXNode(c, Basis.Z)),
        }
    )
)

In [6]:
conditional_observable = ConditionalCorrelationSurface(
    conditions=(_cond,),
    resolutions={
        (False,): _obs_off,
        (True,): _obs_on,
    },
)

In [7]:
cg = compile_block_graph(g, observables=[conditional_observable])
text = cg.generate_conditional_stim_text(k=1)
print(text)

/var/folders/wy/mn07spw97m11pcx4sz_1vv0r0000gn/T/ipykernel_42586/2405585994.py:1: UserWarning: ConditionalCorrelationSurface: skipping per-branch surface validation against substituted BlockGraph. TODO: replace the conditional cube with each branch's ZXCube kind and run _check_correlation_surface_validity on each branch.
  cg = compile_block_graph(g, observables=[conditional_observable])
/Users/isaac/Documents/entropica/tqec_ccf/src/tqec/compile/graph.py:681: UserWarning: resolve_surface_condition_recs: ConditionalAbstractObservable bit 0 (anchor_z=2) resolved to no measurement records in any leaf at z<2. Falling back to placeholder rec[-1]; the IF branch selection is meaningless.
  recs = resolve_surface_condition_recs(


QUBIT_COORDS(0, 0) 0
QUBIT_COORDS(0, 2) 1
QUBIT_COORDS(0, 4) 2
QUBIT_COORDS(0, 6) 3
QUBIT_COORDS(1, 1) 4
QUBIT_COORDS(1, 3) 5
QUBIT_COORDS(1, 5) 6
QUBIT_COORDS(2, 0) 7
QUBIT_COORDS(2, 2) 8
QUBIT_COORDS(2, 4) 9
QUBIT_COORDS(2, 6) 10
QUBIT_COORDS(3, 1) 11
QUBIT_COORDS(3, 3) 12
QUBIT_COORDS(3, 5) 13
QUBIT_COORDS(4, 0) 14
QUBIT_COORDS(4, 2) 15
QUBIT_COORDS(4, 4) 16
QUBIT_COORDS(4, 6) 17
QUBIT_COORDS(5, 1) 18
QUBIT_COORDS(5, 3) 19
QUBIT_COORDS(5, 5) 20
QUBIT_COORDS(6, 0) 21
QUBIT_COORDS(6, 2) 22
QUBIT_COORDS(6, 4) 23
QUBIT_COORDS(6, 6) 24
QUBIT_COORDS(7, 1) 25
QUBIT_COORDS(7, 3) 26
QUBIT_COORDS(7, 5) 27
QUBIT_COORDS(8, 0) 28
QUBIT_COORDS(8, 2) 29
QUBIT_COORDS(8, 4) 30
QUBIT_COORDS(8, 6) 31
QUBIT_COORDS(9, 1) 32
QUBIT_COORDS(9, 3) 33
QUBIT_COORDS(9, 5) 34
QUBIT_COORDS(10, 0) 35
QUBIT_COORDS(10, 2) 36
QUBIT_COORDS(10, 4) 37
QUBIT_COORDS(10, 6) 38
QUBIT_COORDS(11, 1) 39
QUBIT_COORDS(11, 3) 40
QUBIT_COORDS(11, 5) 41
QUBIT_COORDS(12, 0) 42
QUBIT_COORDS(12, 2) 43
QUBIT_COORDS(12, 4) 44
QUBIT_COOR

In [8]:
import re

if_then = re.findall(r"IF\(.*?\) \{\n(.*?)\n\}(?!\s*ELSE)", text, re.DOTALL)
obs_ifs = [body for body in if_then if "OBSERVABLE_INCLUDE(0)" in body]
print(f"# OBSERVABLE_INCLUDE(0) IF blocks (no ELSE): {len(obs_ifs)}")
print(f"# total OBSERVABLE_INCLUDE(0) occurrences: {text.count('OBSERVABLE_INCLUDE(0)')}")

# OBSERVABLE_INCLUDE(0) IF blocks (no ELSE): 1
# total OBSERVABLE_INCLUDE(0) occurrences: 2
